Creating a connection to the Event hub

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Connection string WITH EntityPath
EH_CONN_STR = "ConnectingStringWithEntityPath"

KAFKA_OPTIONS = {
    "kafka.bootstrap.servers": "clickstream.servicebus.windows.net:9093",
    "subscribe": "clickstream",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{EH_CONN_STR}";',
    "startingOffsets": "earliest",
    "kafka.request.timeout.ms": "60000",
    "kafka.session.timeout.ms": "60000"
}



Reading the stream and loading to a bronze layer

In [0]:
KAFKA_OPTIONS.update({
    "startingOffsets": "earliest",
    "minPartitions": "20",
    "maxOffsetsPerTrigger": "100000",
})

df = spark.readStream \
    .format("kafka") \
    .options(**KAFKA_OPTIONS) \
    .option("failOnDataLoss", "false") \
    .load()


query = df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://bronze@clickstreamworkspace.dfs.core.windows.net/checkpoint/") \
    .trigger(availableNow=True) \
    .start("abfss://bronze@clickstreamworkspace.dfs.core.windows.net/Clickstream-Events")

query.awaitTermination()

JSON schema for the data sitting on the bronze layer

In [0]:
json_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("session_id", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("browser", StringType(), True),
    StructField("os", StringType(), True),
    StructField("country", StringType(), True),
    StructField("referrer", StringType(), True),
    StructField("page_url", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("search_term", StringType(), True),
    StructField("cart_value", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("revenue", StringType(), True),
    StructField("error_code", StringType(), True)
])

Loading data from the bronze layer into silver as structured data

In [0]:
silver_df = spark.read.format("delta") \
    .load("abfss://bronze@clickstreamworkspace.dfs.core.windows.net/Clickstream-Events") \
    .withColumn("value", col("value").cast("string")) \
    .withColumn("value", from_json(col("value"), json_schema)) \
    .withColumn("key", col("key").cast("string")) \
    .select("key", "value.*")
    
spark.sql("""
    MERGE INTO silver.events_stage AS t
    USING {source} AS s
    ON t.event_id = s.event_id

    WHEN NOT MATCHED THEN
        INSERT *
""", source=silver_df).show()

Create Tables if they exists for Dimensions

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold.dim_category (
  Category_id      BIGINT GENERATED ALWAYS AS IDENTITY,
  Category         STRING NOT NULL,
  Effective_From   TIMESTAMP,
  Effective_To     TIMESTAMP,
  Is_Current       BOOLEAN,
  Loaded_At        TIMESTAMP,
  CONSTRAINT pk_dim_category PRIMARY KEY (Category_id) RELY
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

CREATE TABLE IF NOT EXISTS gold.dim_product (
  Product_id       BIGINT NOT NULL,
  Product_Name     STRING NOT NULL,
  Category_id      BIGINT,
  Effective_From   TIMESTAMP,
  Effective_To     TIMESTAMP,
  Is_Current       BOOLEAN,
  Loaded_At        TIMESTAMP,
  CONSTRAINT pk_dim_product PRIMARY KEY (Product_id) RELY,
  CONSTRAINT fk_catgory FOREIGN KEY (Category_id) REFERENCES gold.dim_category (Category_id)
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

CREATE TABLE IF NOT EXISTS gold.dim_browser (
  Browser_id         BIGINT GENERATED ALWAYS AS IDENTITY,
  Browser            STRING NOT NULL,
  Effective_From     TIMESTAMP,
  Effective_To       TIMESTAMP,
  Is_Current         BOOLEAN,
  Loaded_At          TIMESTAMP,
  CONSTRAINT pk_dim_browser PRIMARY KEY (Browser_id) RELY
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',       
  'delta.autoOptimize.autoCompact' = 'true'
);

CREATE TABLE IF NOT EXISTS gold.dim_os (
  OS_id            BIGINT GENERATED ALWAYS AS IDENTITY,
  OS               STRING NOT NULL,
  Effective_From   TIMESTAMP,
  Effective_To     TIMESTAMP,
  Is_Current       BOOLEAN,
  Loaded_At        TIMESTAMP,
  CONSTRAINT pk_dim_os PRIMARY KEY (OS_id) RELY
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

CREATE TABLE IF NOT EXISTS gold.dim_country (
  Country_id       BIGINT GENERATED ALWAYS AS IDENTITY,
  Country          STRING NOT NULL,
  Effective_From   TIMESTAMP,
  Effective_To     TIMESTAMP,
  Is_Current       BOOLEAN,
  Loaded_At        TIMESTAMP,
  CONSTRAINT pk_dim_country PRIMARY KEY (Country_id) RELY
) 
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

CREATE TABLE IF NOT EXISTS gold.dim_referrer (
  Referrer_id      BIGINT GENERATED ALWAYS AS IDENTITY,
  Referrer         STRING NOT NULL,
  Effective_From   TIMESTAMP,
  Effective_To     TIMESTAMP,
  Is_Current       BOOLEAN,
  Loaded_At        TIMESTAMP,
  CONSTRAINT pk_dim_referrer PRIMARY KEY (Referrer_id) RELY
) 
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

CREATE TABLE IF NOT EXISTS gold.dim_device (
  Device_id        BIGINT GENERATED ALWAYS AS IDENTITY,
  Device           STRING NOT NULL,
  Effective_From   TIMESTAMP,
  Effective_To     TIMESTAMP,
  Is_Current       BOOLEAN,
  Loaded_At        TIMESTAMP,
  CONSTRAINT pk_dim_device PRIMARY KEY (Device_id) RELY
) 
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
); 

CREATE TABLE IF NOT EXISTS gold.dim_date (
  Date_Key         BIGINT,
  Date             DATE NOT NULL,
  Year             INT,
  Month            INT,
  Day              INT,
CONSTRAINT pk_dim_date PRIMARY KEY (Date_Key) RELY
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
)

Category Dimension

In [0]:
spark.sql("""
    INSERT INTO gold.dim_category(
        Category,
        Effective_From,
        Effective_To,
        Is_Current,
        Loaded_At
    )
    SELECT DISTINCT
        s.category,
        current_timestamp(),
        to_timestamp('9999-12-31 23:59:59', 'yyyy-MM-dd HH:mm:ss'),
        true,
        current_timestamp()
    FROM silver.events_stage s
    WHERE s.category IS NOT NULL
    AND NOT EXISTS (
        SELECT 1 FROM gold.dim_category t
        WHERE t.category = s.category
    )
""").show()

Product Dimension

In [0]:
spark.sql("""
    MERGE INTO gold.dim_product AS t
    USING (
        SELECT DISTINCT RIGHT(product_id,2) AS product_id, product_name, c.category_id
        FROM silver.events_stage s
        INNER JOIN gold.dim_category c ON s.category = c.category
        WHERE s.product_id IS NOT NULL
        ORDER BY product_id
    ) AS s
    ON RIGHT(s.product_id,2) = t.product_id AND t.is_current = true

    WHEN MATCHED AND NOT (t.category_id <=> s.category_id)
    THEN UPDATE SET
        t.effective_to = current_timestamp(),
        t.is_current   = false

    WHEN NOT MATCHED THEN
        INSERT (Product_id, Product_Name, Category_Id, Effective_From, Effective_To, Is_Current, Loaded_At) 
        VALUES (RIGHT(s.product_id,2), s.product_name, s.category_id, current_timestamp(), to_timestamp('9999-12-31 23:59:59', 'yyyy-MM-dd HH:mm:ss'), true, current_timestamp())
""").show()

Browser Dimension

In [0]:
spark.sql("""
    INSERT INTO gold.dim_browser(
        Browser,
        Effective_From,
        Effective_To,
        Is_Current,
        Loaded_At
    )
    SELECT DISTINCT
        s.browser,
        current_timestamp(),
        to_timestamp('9999-12-31 23:59:59', 'yyyy-MM-dd HH:mm:ss'),
        true,
        current_timestamp()
    FROM silver.events_stage s
    WHERE s.browser IS NOT NULL
    AND NOT EXISTS (
        SELECT 1 FROM gold.dim_browser t
        WHERE t.browser = s.browser
    )
""").show()

Operating System Dimension

In [0]:
spark.sql("""
    INSERT INTO gold.dim_os(
        OS,
        Effective_From,
        Effective_To,
        Is_Current,
        Loaded_At
    )
    SELECT DISTINCT
        s.OS,
        current_timestamp(),
        to_timestamp('9999-12-31 23:59:59', 'yyyy-MM-dd HH:mm:ss'),
        true,
        current_timestamp()
    FROM silver.events_stage s
    WHERE s.OS IS NOT NULL
    AND NOT EXISTS (
        SELECT 1 FROM gold.dim_OS t
        WHERE t.OS = s.OS
    )
""").show()

Country Dimenstion

In [0]:
spark.sql("""
    INSERT INTO gold.dim_country(
        Country,
        Effective_From,
        Effective_To,
        Is_Current,
        Loaded_At
    )
    SELECT DISTINCT
        s.Country,
        current_timestamp(),
        to_timestamp('9999-12-31 23:59:59', 'yyyy-MM-dd HH:mm:ss'),
        true,
        current_timestamp()
    FROM silver.events_stage s
    WHERE s.country IS NOT NULL
    AND NOT EXISTS (
        SELECT 1 FROM gold.dim_country t
        WHERE t.Country = s.Country
    )
""").show()

Deivce Dimenstion

In [0]:
spark.sql("""
    INSERT INTO gold.dim_device(
        Device,
        Effective_From,
        Effective_To,
        Is_Current,
        Loaded_At
    )
    SELECT 
        s.device_type,
        current_timestamp(),
        to_timestamp('9999-12-31 23:59:59', 'yyyy-MM-dd HH:mm:ss'),
        true,
        current_timestamp()
    FROM silver.events_stage s
    WHERE s.device_type IS NOT NULL
    AND NOT EXISTS (
        SELECT 1 FROM gold.dim_device t
        WHERE t.Device = s.device_type
    )
""").show()

Refereral Dimension

In [0]:
spark.sql("""
    INSERT INTO gold.dim_referrer(
        Referrer,
        Effective_From,
        Effective_To,
        Is_Current,
        Loaded_At
    )
    SELECT 
        s.Referrer,
        current_timestamp(),
        to_timestamp('9999-12-31 23:59:59', 'yyyy-MM-dd HH:mm:ss'),
        true,
        current_timestamp()
    FROM silver.events_stage s
    WHERE s.Referrer IS NOT NULL
    AND NOT EXISTS (
        SELECT 1 FROM gold.dim_referrer t
        WHERE t.Referrer = s.Referrer
    )
""").show()

Date Dimension

In [0]:
dim_date = (
    silver_df
    .select(to_date("event_timestamp").alias("date"))
    .dropDuplicates()
    .withColumn("date_key", date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", year("date"))
    .withColumn("month", month("date"))
    .withColumn("day", dayofmonth("date"))
)
# Loading data
spark.sql("""
    INSERT INTO gold.dim_date(
        Date_Key,
        Date,
        Year,
        Month,
        Day
    )
    SELECT
        s.date_key,
        s.date,
        s.year,
        s.month,
        s.day
    FROM {source} s
    WHERE NOT EXISTS (
        SELECT 1 FROM gold.dim_date t
        WHERE t.Date_Key = s.date_key
    )
""", source=dim_date).show()

In [0]:
dim_date = spark.sql("""
    INSERT INTO gold.dim_date(
        Date_Key,
        Date,
        Year,
        Month,
        Day
    )
    SELECT DISTINCT
        CAST(date_format(s.event_timestamp, "yyyyMMdd") AS int),
        CAST(date_format(s.event_timestamp, "yyyy-MM-dd") AS date),
        year(s.event_timestamp),
        month(s.event_timestamp),
        day(s.event_timestamp)
    FROM silver.events_stage s
    WHERE s.event_timestamp IS NOT NULL
    AND NOT EXISTS (
        SELECT 1 FROM gold.dim_date t
        WHERE t.Date_Key = CAST(date_format(s.event_timestamp, "yyyyMMdd") AS int)
    )          
""").show()

Create Tables if they exists for Facts

In [0]:
%sql

CREATE TABLE IF NOT EXISTS gold.fact_events(
  Event_id         STRING NOT NULL,
  Event_type       STRING NOT NULL,
  Date_Key         BIGINT,
  User_id          STRING NOT NULL,
  Session_id       STRING NOT NULL,
  Product_id       BIGINT,
  Device_id        BIGINT,
  Browser_id       BIGINT,
  OS_id            BIGINT,
  Referrer_id      BIGINT,
  Country_id       BIGINT,
  Cart_value       DECIMAL(12,2),
  Revenue          DECIMAL(12,2),
  CONSTRAINT fk_event_product FOREIGN KEY (Product_id) REFERENCES gold.dim_product (Product_id),
  CONSTRAINT fk_event_browser FOREIGN KEY (Browser_id) REFERENCES gold.dim_browser (Browser_id),
  CONSTRAINT fk_event_os FOREIGN KEY (OS_id) REFERENCES gold.dim_os (OS_id),
  CONSTRAINT fk_event_country FOREIGN KEY (Country_id) REFERENCES gold.dim_country (Country_id),
  CONSTRAINT fk_event_device FOREIGN KEY (Device_id) REFERENCES gold.dim_device (Device_id),
  CONSTRAINT fk_event_referrer FOREIGN KEY (Referrer_id) REFERENCES gold.dim_referrer (Referrer_id),
  CONSTRAINT fk_event_date FOREIGN KEY (Date_Key) REFERENCES gold.dim_date (Date_Key),
  CONSTRAINT pk_events PRIMARY KEY (Event_id) RELY
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

CREATE TABLE IF NOT EXISTS gold.fact_orders (
  order_id         STRING NOT NULL,
  Event_type       STRING NOT NULL,
  Date_Key         BIGINT,
  User_id          STRING NOT NULL,
  Session_id       STRING NOT NULL,
  Product_id       BIGINT,
  Revenue          DECIMAL(12,2),
  CONSTRAINT fk_orders_product FOREIGN KEY (Product_id) REFERENCES gold.dim_product (Product_id),
  CONSTRAINT pk_orders PRIMARY KEY (order_id) RELY
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);


CREATE TABLE IF NOT EXISTS gold.fact_sessions(
  Session_id       STRING NOT NULL,
  User_id          STRING NOT NULL,
  Session_start_timestamp TIMESTAMP,
  Session_end_timestamp TIMESTAMP,
  Session_duration_seconds BIGINT,
  Device_id        BIGINT,
  Browser_id       BIGINT,
  OS_id            BIGINT,
  Referrer_id      BIGINT,
  Country_id       BIGINT,
  Page_views       BIGINT,
  Product_views    BIGINT,
  Add_to_cart_count BIGINT,
  Checkout_started_flag BOOLEAN,
  Purchase_flag    BOOLEAN,
  Session_revenue  DECIMAL(12,2),
  CONSTRAINT fk_sessions_device FOREIGN KEY (Device_id) REFERENCES gold.dim_device (Device_id),
  CONSTRAINT fk_sessions_browser FOREIGN KEY (Browser_id) REFERENCES gold.dim_browser (Browser_id),
  CONSTRAINT fk_sessions_os FOREIGN KEY (OS_id) REFERENCES gold.dim_os (OS_id),
  CONSTRAINT fk_sessions_country FOREIGN KEY (Country_id) REFERENCES gold.dim_country (Country_id),
  CONSTRAINT fk_sessions_referrer FOREIGN KEY (Referrer_id) REFERENCES gold.dim_referrer (Referrer_id),
  CONSTRAINT pk_sessions PRIMARY KEY (Session_id) RELY
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

Fact Events

In [0]:
spark.sql("""
    INSERT INTO gold.fact_events (
        Event_id,
        Event_type,
        Date_Key,
        User_id,
        Session_id,
        Product_id,
        Device_id,
        Browser_id,
        OS_id,
        Referrer_id,
        Country_id,
        Cart_value,
        Revenue
    )
    SELECT DISTINCT
        S.event_id,
        S.event_type,
        d.Date_Key,
        S.user_id,
        S.session_id,
        COALESCE(p.Product_id,0) AS Product_id,
        d.Device_id,
        br.Browser_id,
        o.OS_id,
        r.referrer_id,
        c.Country_id,
        COALESCE(CAST(s.cart_value AS DECIMAL(10,2)), 0) AS cart_value,
        COALESCE(CAST(s.revenue AS DECIMAL(10,2)), 0) AS revenue
    FROM silver.events_stage s
    INNER JOIN gold.dim_date d
    ON date_format(s.event_timestamp, "yyyyMMdd") = d.Date_Key
    LEFT OUTER JOIN gold.dim_product p
    ON CAST(RIGHT(s.product_id,2) AS INT) = p.Product_id
    LEFT OUTER JOIN gold.dim_os o
    ON s.os = o.OS
    LEFT OUTER JOIN gold.dim_country c
    ON s.country = c.Country
    LEFT OUTER JOIN gold.dim_referrer r
    ON s.referrer = r.Referrer
    LEFT OUTER JOIN gold.dim_browser br
    ON s.browser = br.Browser
    LEFT OUTER JOIN gold.dim_device d
    ON s.device_type = d.Device
    WHERE NOT EXISTS (
        SELECT 1 FROM gold.fact_events e
        WHERE e.Event_id = s.event_id
    )
""").show()

Fact Orders

In [0]:
spark.sql("""
    INSERT INTO gold.fact_orders (
        order_id,
        Event_type,
        Date_Key,
        User_id,
        Session_id,
        Product_id,
        Revenue
    )
    SELECT DISTINCT
        S.order_id,
        S.event_type,
        d.Date_Key,
        S.user_id,
        S.session_id,
        COALESCE(p.Product_id,0) AS Product_id,
        COALESCE(CAST(s.revenue AS DECIMAL(10,2)), 0) AS revenue
    FROM silver.events_stage s
    INNER JOIN gold.dim_date d
    ON date_format(s.event_timestamp, "yyyyMMdd") = d.Date_Key
    LEFT OUTER JOIN gold.dim_product p
    ON CAST(RIGHT(s.product_id,2) AS INT) = p.Product_id
    LEFT OUTER JOIN gold.dim_os o
    ON s.os = o.OS
    LEFT OUTER JOIN gold.dim_country c
    ON s.country = c.Country
    LEFT OUTER JOIN gold.dim_referrer r
    ON s.referrer = r.Referrer
    LEFT OUTER JOIN gold.dim_browser br
    ON s.browser = br.Browser
    LEFT OUTER JOIN gold.dim_device d
    ON s.device_type = d.Device
    WHERE s.event_type = 'purchase'
    AND NOT EXISTS (
        SELECT 1 FROM gold.fact_orders o
        WHERE o.order_id = s.order_id
    )
""").show()

Fact Sessions

In [0]:
spark.sql("""
    INSERT INTO gold.fact_sessions (
        session_id,
        user_id,
        session_start_timestamp,
        session_end_timestamp,
        session_duration_seconds,
        Device_id,
        Browser_id,
        OS_id,
        referrer_id,
        country_id,
        page_views,
        product_views,
        add_to_cart_count,
        checkout_started_flag,
        purchase_flag,
        session_revenue
    )
    SELECT DISTINCT
        S.session_id,
        MIN(s.user_id) AS user_id,
        MIN(s.event_timestamp) AS session_start_timestamp,
        MAX(s.event_timestamp) AS session_end_timestamp,
        DATEDIFF(second, MIN(event_timestamp), MAX(event_timestamp)) AS session_duration_seconds,
        MIN(d.Device_id) AS Device_id,
        MIN(br.Browser_id) AS Browser_id,
        MIN(o.OS_id) AS OS_id,
        MIN(r.referrer_id) AS referrer_id,
        MIN(c.country_id) AS country_id,
        SUM(CASE WHEN s.event_type = 'page_view' THEN 1 ELSE 0 END) AS page_views,
        SUM(CASE WHEN s.event_type = 'product_view' THEN 1 ELSE 0 END) AS product_views,
        SUM(CASE WHEN s.event_type = 'add_to_cart' THEN 1 ELSE 0 END) AS add_to_cart_count,
        MAX(CASE WHEN s.event_type = 'checkout_start' THEN 1 ELSE 0 END) AS checkout_started_flag,
        MAX(CASE WHEN s.event_type = 'purchase' THEN 1 ELSE 0 END) AS purchase_flag,
        SUM(CASE WHEN s.event_type = 'purchase' THEN CAST(s.revenue AS DECIMAL(10,2)) ELSE 0 END) AS session_revenue
    FROM silver.events_stage s
    INNER JOIN gold.dim_date d
    ON date_format(s.event_timestamp, "yyyyMMdd") = d.Date_Key
    LEFT OUTER JOIN gold.dim_product p
    ON CAST(RIGHT(s.product_id,2) AS INT) = p.Product_id
    LEFT OUTER JOIN gold.dim_os o
    ON s.os = o.OS
    LEFT OUTER JOIN gold.dim_country c
    ON s.country = c.Country
    LEFT OUTER JOIN gold.dim_referrer r
    ON s.referrer = r.Referrer
    LEFT OUTER JOIN gold.dim_browser br
    ON s.browser = br.Browser
    LEFT OUTER JOIN gold.dim_device d
    ON s.device_type = d.Device
    WHERE NOT EXISTS (        
        SELECT 1 FROM gold.fact_sessions o
        WHERE o.session_id = s.session_id
    )
    GROUP BY S.session_id
""").show()